# Phase 3 & 4 — Figure Generation for LaTeX Documentation

This notebook generates all six figures referenced in `PHASE3_4_REPORT.tex`:
1. `dataset_split.png` — Chronological train/val/test split
2. `modality_coverage.png` — Modality availability chart
3. `volatility_analysis.png` — Volatility distributions and clustering
4. `architecture.png` — Model architecture diagram
5. `r2_progression.png` — R² improvement across phases
6. `gate_weights.png` — Learned fusion gate weights

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.dates as mdates
from matplotlib.gridspec import GridSpec
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# ── Style ──
plt.rcParams.update({
    'figure.dpi': 200,
    'savefig.dpi': 200,
    'font.family': 'sans-serif',
    'font.size': 11,
    'axes.titlesize': 14,
    'axes.titleweight': 'bold',
    'axes.labelsize': 12,
    'figure.facecolor': 'white',
})

# ── Paths ──
DATA_DIR = Path('../data')
PRICE_DIR = DATA_DIR / 'raw' / 'prices'
TARGET_DIR = DATA_DIR / 'targets'
FIG_DIR = Path('../docs/figures')
FIG_DIR.mkdir(parents=True, exist_ok=True)
print(f'Saving figures to: {FIG_DIR.resolve()}')

---
## Figure 1: `dataset_split.png`

In [ ]:
# Load all price data to get date range
all_dates = []
price_files = sorted(PRICE_DIR.glob('*_ohlcv.csv'))
for fp in price_files:
    df = pd.read_csv(fp, parse_dates=['date'])
    if 'tech10' in fp.stem:
        continue
    all_dates.extend(df['date'].tolist())

all_dates = pd.Series(all_dates)
all_dates = pd.to_datetime(all_dates)

# Define split boundaries (matching SplitConfig in preprocessing.py)
train_end = pd.Timestamp('2022-12-31')
val_end   = pd.Timestamp('2023-12-31')

train_dates = all_dates[all_dates <= train_end]
val_dates   = all_dates[(all_dates > train_end) & (all_dates <= val_end)]
test_dates  = all_dates[all_dates > val_end]

print(f'Train: {len(train_dates):,} samples ({len(train_dates)/len(all_dates):.1%})')
print(f'Val:   {len(val_dates):,} samples ({len(val_dates)/len(all_dates):.1%})')
print(f'Test:  {len(test_dates):,} samples ({len(test_dates)/len(all_dates):.1%})')

fig, ax = plt.subplots(figsize=(12, 3.5))

colors = {'Train': '#3498db', 'Validation': '#f39c12', 'Test': '#e74c3c'}

# Count samples per month per split
for name, dates, color in [
    ('Train', train_dates, colors['Train']),
    ('Validation', val_dates, colors['Validation']),
    ('Test', test_dates, colors['Test']),
]:
    monthly = dates.dt.to_period('M').value_counts().sort_index()
    months = monthly.index.to_timestamp()
    ax.bar(months, monthly.values, width=25, color=color, alpha=0.85, label=name, edgecolor='white', linewidth=0.3)

ax.axvline(train_end, color='gray', linestyle='--', linewidth=1.2, alpha=0.7)
ax.axvline(val_end, color='gray', linestyle='--', linewidth=1.2, alpha=0.7)
ax.text(train_end - pd.Timedelta(days=180), ax.get_ylim()[1]*0.9, 'Train End', fontsize=9, color='gray')
ax.text(val_end - pd.Timedelta(days=150), ax.get_ylim()[1]*0.9, 'Val End', fontsize=9, color='gray')

ax.set_title('Chronological Train / Validation / Test Split')
ax.set_xlabel('Date')
ax.set_ylabel('Samples per Month')
ax.legend(loc='upper left', fontsize=10)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
plt.tight_layout()
fig.savefig(FIG_DIR / 'dataset_split.png', bbox_inches='tight')
plt.show()
print('Saved: dataset_split.png')

---
## Figure 2: `modality_coverage.png`

In [ ]:
# Modality coverage data (from project stats)
modalities = ['Price\n(OHLCV)', 'Graph\n(GAT)', 'Macro\nIndicators', 'SEC 10-K\nFilings']
coverage = [100.0, 99.3, 99.6, 73.7]
sample_counts = [73793, 73279, 73498, 54406]
bar_colors = ['#3498db', '#2ecc71', '#9b59b6', '#e67e22']

fig, ax = plt.subplots(figsize=(10, 5))

bars = ax.bar(modalities, coverage, color=bar_colors, edgecolor='white', width=0.6, alpha=0.9)

for bar, pct, count in zip(bars, coverage, sample_counts):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1.2,
            f'{pct:.1f}%\n({count:,})', ha='center', fontsize=10, fontweight='bold')

ax.set_ylim(0, 115)
ax.set_title('Modality Coverage Across 73,793 Samples')
ax.set_ylabel('Coverage (%)')
ax.axhline(100, color='gray', linestyle=':', alpha=0.4)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
fig.savefig(FIG_DIR / 'modality_coverage.png', bbox_inches='tight')
plt.show()
print('Saved: modality_coverage.png')

---
## Figure 3: `volatility_analysis.png`

In [ ]:
# Load volatility targets
vol_df = pd.read_csv(TARGET_DIR / 'volatility_targets.csv', parse_dates=['date'])

fig, axes = plt.subplots(1, 2, figsize=(16, 5.5))

# ── Left: Boxplot by ticker ──
select_tickers = ['AAPL', 'MSFT', 'GOOGL', 'NVDA', 'TSLA', 'META', 'AMD', 'AMZN']
subset = vol_df[vol_df['ticker'].isin(select_tickers)]

# Order by median volatility
medians = subset.groupby('ticker')['realized_vol_20d_annualized'].median().sort_values()
ordered_tickers = medians.index.tolist()

box_data = [subset[subset['ticker'] == t]['realized_vol_20d_annualized'].dropna().values for t in ordered_tickers]
bp = axes[0].boxplot(box_data, labels=ordered_tickers, patch_artist=True,
                      showfliers=False, widths=0.6,
                      medianprops=dict(color='black', linewidth=1.5))

cmap = plt.cm.coolwarm
for i, patch in enumerate(bp['boxes']):
    patch.set_facecolor(cmap(i / len(bp['boxes'])))
    patch.set_alpha(0.75)

axes[0].set_title('Volatility Distribution by Stock')
axes[0].set_ylabel('Annualized Realized Volatility')
axes[0].set_xlabel('Ticker (ordered by median)')
axes[0].tick_params(axis='x', rotation=45)

# ── Right: Volatility clustering over time ──
for ticker, color in zip(['AAPL', 'TSLA', 'NVDA', 'MSFT'],
                          ['#3498db', '#e74c3c', '#2ecc71', '#f39c12']):
    ts = vol_df[vol_df['ticker'] == ticker].sort_values('date')
    axes[1].plot(ts['date'], ts['realized_vol_20d_annualized'],
                label=ticker, linewidth=0.7, alpha=0.85, color=color)

axes[1].set_title('Volatility Clustering Over Time')
axes[1].set_xlabel('Date')
axes[1].set_ylabel('Annualized Volatility')
axes[1].legend(fontsize=9, loc='upper right')
axes[1].xaxis.set_major_formatter(mdates.DateFormatter('%Y'))

plt.tight_layout()
fig.savefig(FIG_DIR / 'volatility_analysis.png', bbox_inches='tight')
plt.show()
print('Saved: volatility_analysis.png')

---
## Figure 4: `architecture.png`

In [ ]:
fig, ax = plt.subplots(figsize=(16, 9))
ax.set_xlim(0, 16)
ax.set_ylim(0, 9)
ax.axis('off')

def draw_box(ax, x, y, w, h, text, color='#3498db', fontsize=9, text_color='white'):
    rect = mpatches.FancyBboxPatch((x, y), w, h, boxstyle='round,pad=0.1',
                                    facecolor=color, edgecolor='white', linewidth=1.5)
    ax.add_patch(rect)
    ax.text(x + w/2, y + h/2, text, ha='center', va='center',
            fontsize=fontsize, fontweight='bold', color=text_color)

def draw_arrow(ax, x1, y1, x2, y2, color='#555', lw=1.5, style='->'):
    ax.annotate('', xy=(x2, y2), xytext=(x1, y1),
                arrowprops=dict(arrowstyle=style, color=color, lw=lw))

# ── Title ──
ax.text(8, 8.6, 'Multimodal Volatility & Direction Forecasting Architecture',
        ha='center', fontsize=15, fontweight='bold', color='#1d3557')

# ── Row 1: Data Sources ──
data_y = 7.5
sources = [
    (0.5,  'Price Data\n[B, 60, 21]', '#2c3e50'),
    (3.6,  'Sector Graph\n30 nodes, 200 edges', '#2c3e50'),
    (6.7,  'SEC 10-K Filings\n227 documents', '#2c3e50'),
    (9.8,  'Macro Indicators\n[B, 12]', '#2c3e50'),
    (12.9, 'HAR-RV Features\n[B, 3]', '#c0392b'),
]
for x, text, color in sources:
    draw_box(ax, x, data_y, 2.6, 0.8, text, color, fontsize=8)

# ── Row 2: Encoders ──
enc_y = 5.8
encoders = [
    (0.5,  'CNN-BiLSTM\nEncoder', '#2980b9'),
    (3.6,  'Graph Attention\nNetwork (GAT)', '#27ae60'),
    (6.7,  'FinBERT +\nAttn Pooling', '#8e44ad'),
    (9.8,  'Macro MLP\nEncoder', '#d35400'),
]
for x, text, color in encoders:
    draw_box(ax, x, enc_y, 2.6, 0.8, text, color, fontsize=8)

# Arrows: Data -> Encoders
for i in range(4):
    sx = sources[i][0] + 1.3
    draw_arrow(ax, sx, data_y, sx, enc_y + 0.8)

# ── Row 3: Embeddings ──
emb_y = 4.5
embeddings = [
    (0.5,  '[B, 256]', '#5dade2'),
    (3.6,  '[B, 256]', '#58d68d'),
    (6.7,  '[B, 768]', '#af7ac5'),
    (9.8,  '[B, 32]',  '#eb984e'),
]
for x, text, color in embeddings:
    draw_box(ax, x, emb_y, 2.6, 0.45, text, color, fontsize=9)

# Arrows: Encoders -> Embeddings
for i in range(4):
    sx = encoders[i][0] + 1.3
    draw_arrow(ax, sx, enc_y, sx, emb_y + 0.45)

# ── HAR-RV Skip Connection (orange arrow) ──
har_x = sources[4][0] + 1.3
draw_box(ax, 12.9, emb_y, 2.6, 0.45, '[B, 32]', '#e74c3c', fontsize=9)
draw_arrow(ax, har_x, data_y, har_x, emb_y + 0.45, color='#e74c3c', lw=2.5)
ax.text(har_x + 0.3, (data_y + emb_y + 0.45)/2, 'Skip\nConnection',
        fontsize=7, color='#e74c3c', fontstyle='italic')

# ── Gating ──
gate_y = 3.3
draw_box(ax, 2.5, gate_y, 8.0, 0.7,
         'Sigmoid Gated Attention Fusion\n(conditioned on earnings surprise features)',
         '#1abc9c', fontsize=9)

# Arrows: Embeddings -> Gate
for i in range(4):
    sx = embeddings[i][0] + 1.3
    draw_arrow(ax, sx, emb_y, 6.5, gate_y + 0.7)

# ── Shared Trunk ──
trunk_y = 2.0
draw_box(ax, 3.5, trunk_y, 9.0, 0.7,
         'Shared Trunk (2-layer MLP + LayerNorm + GELU + Dropout)',
         '#34495e', fontsize=9)

# Arrow: Gate -> Trunk (gated modalities + HAR concatenated)
draw_arrow(ax, 6.5, gate_y, 8.0, trunk_y + 0.7)
# Arrow: HAR skip -> Trunk
draw_arrow(ax, har_x, emb_y, 12.0, trunk_y + 0.7, color='#e74c3c', lw=2.5)
ax.text(12.7, (emb_y + trunk_y + 0.7)/2 + 0.2, 'concat',
        fontsize=7, color='#e74c3c', fontstyle='italic')

# ── Task Heads ──
head_y = 0.5
draw_box(ax, 3.0, head_y, 4.0, 0.8,
         'Volatility Head (Softplus)\nPRIMARY — QLIKE Loss (0.85)',
         '#c0392b', fontsize=9)
draw_box(ax, 8.5, head_y, 4.0, 0.8,
         'Direction Head\nAUXILIARY — ListNet Loss (0.15)',
         '#2c3e50', fontsize=9)

# Arrows: Trunk -> Heads
draw_arrow(ax, 6.5, trunk_y, 5.0, head_y + 0.8)
draw_arrow(ax, 9.5, trunk_y, 10.5, head_y + 0.8)

plt.tight_layout()
fig.savefig(FIG_DIR / 'architecture.png', bbox_inches='tight')
plt.show()
print('Saved: architecture.png')

---
## Figure 5: `r2_progression.png`

In [ ]:
# R² progression data from project README
phases = [
    'Historical\nAverage', 'V1 Fusion\n(Phase 6)', 'V2+ListNet\n(Phase 10)',
    'QLIKE+30\n(Phase 12)', 'HAR-RV\n(Phase 13)', 'HAR Skip\n(Phase 14)',
    'HAR-RV\nBenchmark'
]
r2_values = [0.348, 0.415, 0.335, 0.772, 0.867, 0.921, 0.947]
auc_values = [None, 0.516, 0.599, 0.568, 0.585, 0.591, None]

fig, ax1 = plt.subplots(figsize=(14, 5.5))

# R² bars
bar_colors = ['#95a5a6'] + ['#3498db']*5 + ['#2ecc71']
bars = ax1.bar(range(len(phases)), r2_values, color=bar_colors, alpha=0.85,
               edgecolor='white', width=0.6)

# Highlight Phase 14
bars[5].set_facecolor('#e74c3c')
bars[5].set_alpha(0.95)

for i, (bar, val) in enumerate(zip(bars, r2_values)):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.015,
             f'{val:.3f}', ha='center', fontsize=9, fontweight='bold')

ax1.set_xticks(range(len(phases)))
ax1.set_xticklabels(phases, fontsize=9)
ax1.set_ylabel('Volatility R²', fontsize=12)
ax1.set_ylim(0, 1.1)
ax1.set_title('Volatility R² Progression Across Project Phases')

# AUC line (secondary axis)
ax2 = ax1.twinx()
auc_x = [i for i, v in enumerate(auc_values) if v is not None]
auc_y = [v for v in auc_values if v is not None]
ax2.plot(auc_x, auc_y, 'o-', color='#f39c12', linewidth=2, markersize=8,
         label='Direction AUC', zorder=5)
for x, y in zip(auc_x, auc_y):
    ax2.text(x + 0.15, y, f'{y:.3f}', fontsize=8, color='#f39c12')
ax2.set_ylabel('Direction AUC', fontsize=12, color='#f39c12')
ax2.set_ylim(0.45, 0.65)
ax2.tick_params(axis='y', labelcolor='#f39c12')

# HAR-RV benchmark line
ax1.axhline(0.947, color='#2ecc71', linestyle='--', alpha=0.6, linewidth=1.2)
ax1.text(0.5, 0.955, 'HAR-RV Benchmark (R²=0.947)', fontsize=8, color='#2ecc71')

# Arrow annotation for Phase 14
ax1.annotate('Key: HAR-RV\nskip connection',
             xy=(5, 0.921), xytext=(5.8, 0.75),
             fontsize=8, color='#c0392b',
             arrowprops=dict(arrowstyle='->', color='#c0392b', lw=1.5))

ax1.spines['top'].set_visible(False)
plt.tight_layout()
fig.savefig(FIG_DIR / 'r2_progression.png', bbox_inches='tight')
plt.show()
print('Saved: r2_progression.png')

---
## Figure 6: `gate_weights.png`

In [ ]:
# Gate weight data from project results
modalities_gate = ['GAT\n(Graph)', 'Price\n(CNN-BiLSTM)', 'Document\n(FinBERT)', 'Macro\n(MLP)']
weights = [41.2, 30.1, 18.2, 10.5]
gate_colors = ['#2ecc71', '#3498db', '#9b59b6', '#e67e22']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: Bar chart
bars = axes[0].barh(modalities_gate, weights, color=gate_colors, alpha=0.85,
                     edgecolor='white', height=0.55)
for bar, w in zip(bars, weights):
    axes[0].text(bar.get_width() + 0.8, bar.get_y() + bar.get_height()/2,
                 f'{w:.1f}%', va='center', fontsize=11, fontweight='bold')
axes[0].set_xlim(0, 55)
axes[0].set_xlabel('Average Gate Weight (%)')
axes[0].set_title('Learned Modality Gate Weights')
axes[0].invert_yaxis()
axes[0].spines['top'].set_visible(False)
axes[0].spines['right'].set_visible(False)

# Right: Pie chart
wedges, texts, autotexts = axes[1].pie(
    weights, labels=modalities_gate, colors=gate_colors,
    autopct='%1.1f%%', startangle=90,
    pctdistance=0.65, labeldistance=1.15,
    wedgeprops=dict(edgecolor='white', linewidth=2)
)
for autotext in autotexts:
    autotext.set_fontsize(10)
    autotext.set_fontweight('bold')
axes[1].set_title('Gate Weight Distribution')

plt.tight_layout()
fig.savefig(FIG_DIR / 'gate_weights.png', bbox_inches='tight')
plt.show()
print('Saved: gate_weights.png')

---
## Additional: EDA Visualizations

In [ ]:
# ── Direction Label Distribution ──
dir_df = pd.read_csv(TARGET_DIR / 'direction_labels_multi_horizon.csv', parse_dates=['date'])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for i, horizon in enumerate(['5d', '60d']):
    col = f'direction_{horizon}_label'
    counts = dir_df[col].value_counts()
    clrs = ['#2ecc71' if l == 'UP' else '#e74c3c' for l in counts.index]
    axes[i].bar(counts.index, counts.values, color=clrs, edgecolor='white', width=0.4)
    axes[i].set_title(f'{horizon.upper()} Direction Label Distribution')
    axes[i].set_ylabel('Count')
    total = counts.sum()
    for j, (label, count) in enumerate(counts.items()):
        axes[i].text(j, count + total*0.01, f'{count/total:.1%}',
                     ha='center', fontweight='bold', fontsize=11)

plt.tight_layout()
plt.show()

In [ ]:
# ── Normalized Price Trajectories ──
fig, ax = plt.subplots(figsize=(16, 6))

for ticker in ['AAPL', 'MSFT', 'GOOGL', 'NVDA', 'TSLA', 'META', 'AMD', 'AMZN']:
    fp = PRICE_DIR / f'{ticker}_ohlcv.csv'
    if fp.exists():
        df = pd.read_csv(fp, parse_dates=['date']).sort_values('date')
        norm = df['close'] / df['close'].iloc[0] * 100
        ax.plot(df['date'], norm, label=ticker, linewidth=1.0)

ax.set_title('Normalized Price Trajectories (Base = 100)')
ax.set_xlabel('Date')
ax.set_ylabel('Normalized Price')
ax.legend(ncol=4, fontsize=9)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
plt.tight_layout()
plt.show()

In [ ]:
# ── Feature Engineering Demo ──
fp = PRICE_DIR / 'AAPL_ohlcv.csv'
df = pd.read_csv(fp, parse_dates=['date']).sort_values('date')

# Compute features
df['log_return'] = np.log(df['close'] / df['close'].shift(1))
df['rv_daily'] = df['log_return']**2
df['rv_weekly'] = df['rv_daily'].rolling(5).mean()
df['rv_monthly'] = df['rv_daily'].rolling(22).mean()
df = df.dropna()

fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# Return distribution
axes[0,0].hist(df['log_return'], bins=100, color='steelblue', alpha=0.7, edgecolor='white')
axes[0,0].axvline(0, color='red', linestyle='--', alpha=0.7)
axes[0,0].set_title('AAPL — Daily Log Return Distribution')
axes[0,0].set_xlabel('Log Return')
stats_text = f'μ={df["log_return"].mean():.5f}\nσ={df["log_return"].std():.4f}\nSkew={df["log_return"].skew():.2f}\nKurt={df["log_return"].kurtosis():.1f}'
axes[0,0].text(0.02, 0.95, stats_text, transform=axes[0,0].transAxes, fontsize=9,
               va='top', bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

# HAR-RV components
axes[0,1].plot(df['date'], df['rv_daily'], alpha=0.3, linewidth=0.4, color='gray', label='Daily RV')
axes[0,1].plot(df['date'], df['rv_weekly'], alpha=0.8, linewidth=1, color='#3498db', label='Weekly RV (5d)')
axes[0,1].plot(df['date'], df['rv_monthly'], alpha=0.8, linewidth=1.3, color='#e74c3c', label='Monthly RV (22d)')
axes[0,1].set_title('AAPL — HAR-RV Components')
axes[0,1].set_xlabel('Date')
axes[0,1].legend(fontsize=9)
axes[0,1].xaxis.set_major_formatter(mdates.DateFormatter('%Y'))

# Z-score normalization demo
n = len(df)
train = df.iloc[:int(n*0.7)]
mu, sigma = train['log_return'].mean(), train['log_return'].std()
df['log_return_norm'] = (df['log_return'] - mu) / (sigma + 1e-8)

axes[1,0].hist(df.iloc[:int(n*0.7)]['log_return'], bins=80, alpha=0.5, color='#3498db', label='Raw (train)')
axes[1,0].hist(df.iloc[:int(n*0.7)]['log_return_norm'], bins=80, alpha=0.5, color='#e74c3c', label='Z-scored (train)')
axes[1,0].set_title('Z-Score Normalization Effect')
axes[1,0].set_xlabel('Value')
axes[1,0].legend(fontsize=9)

# Volume features
df['volume_sma_20'] = df['volume'].rolling(20).mean()
df['volume_ratio'] = df['volume'] / df['volume_sma_20']
recent = df[df['date'] >= '2024-01-01']
axes[1,1].bar(recent['date'], recent['volume_ratio'], width=1.5, alpha=0.6, color='#9b59b6')
axes[1,1].axhline(1.0, color='black', linestyle='--', alpha=0.5)
axes[1,1].set_title('AAPL — Volume Ratio (Vol / 20d SMA)')
axes[1,1].set_xlabel('Date')
axes[1,1].xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))

plt.tight_layout()
plt.show()

In [ ]:
# ── Cross-Ticker Correlation ──
import seaborn as sns

pivot = pd.DataFrame()
for ticker in ['AAPL', 'MSFT', 'GOOGL', 'NVDA', 'TSLA', 'META', 'AMD', 'AMZN']:
    fp = PRICE_DIR / f'{ticker}_ohlcv.csv'
    if fp.exists():
        df = pd.read_csv(fp, parse_dates=['date']).set_index('date')
        pivot[ticker] = np.log(df['close'] / df['close'].shift(1))

corr = pivot.corr()

fig, ax = plt.subplots(figsize=(9, 7))
mask = np.triu(np.ones_like(corr, dtype=bool), k=1)
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, vmin=-0.2, vmax=0.8, square=True, linewidths=0.5,
            cbar_kws={'shrink': 0.8}, ax=ax)
ax.set_title('Cross-Ticker Log Return Correlations')
plt.tight_layout()
plt.show()

In [ ]:
print('\n' + '='*60)
print('ALL FIGURES GENERATED SUCCESSFULLY')
print('='*60)
print(f'\nFigures saved to: {FIG_DIR.resolve()}')
print('\nFiles:')
for f in sorted(FIG_DIR.glob('*.png')):
    print(f'  ✓ {f.name}')